# 26 - Association study, Stage 0: frame identity and timing audit (CPU)

`paper/cvpr2027/DEPTH_USS_ASSOCIATION_PROPOSAL.md` §13 releases **only** this audit:
no model, no annotation, no association parameter. CPU runtime is enough.

For every clip in the fixed sample `configs/assoc/stage0_sample_v1.csv` (83 non-reserved
sequences, one per recording day plus rig top-ups, 143 clips) it identifies which decoded
video frame each cached window JPEG is (exact JPEG bytes first, else a declared pixel match),
reads its real PTS, maps it to MF4 time through the camera channel, and compares it with the
nominal time notebook 20 assumed. It also logs how often USS map-object and echo values change.

Reads cached frames, videos and MF4s; writes only `ARTIFACTS/assoc/stage0_timing_v2/` (v1 kept: it missed upper-case `.MF4` files).
All 154 reserved IDs are asserted absent before any sequence is opened.

Set `SMOKE = True` first (3 clips into `stage0_timing_v2_smoke`), then `False` for the full audit.
Send back `summary.json`, `clips_audit.csv`, `frames_audit.csv`.


In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    REPO_URL = "https://github.com/Ali-RT/vision-uss-research"
    if not (REPO_DIR / ".git").exists():   # fresh VM: clone (no separate bootstrap script)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    # REPO_REF: "origin/main" (latest) by default; a Step 4 M3 run sets the frozen
    # code SHA from configs/step4/freeze_inventory.json. Discards VM-local edits.
    REPO_REF = "origin/main"
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", REPO_REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)   # PyAV: PTS cross-check

SMOKE = True        # True: first 3 clips -> stage0_timing_v1_smoke; False: full audit
WORKERS = 4

cmd = [sys.executable, str(REPO_DIR / "scripts" / "assoc_stage0_timing.py"), "audit",
       "--sample", str(REPO_DIR / "configs" / "assoc" / "stage0_sample_v1.csv"),
       "--raw-root", str(RAW_ROOT),
       "--frames-root", str(ARTIFACTS / "frames" / "r30-85_n20"),
       "--out", str(ARTIFACTS / "assoc" / "stage0_timing_v2"),
       "--workers", str(WORKERS)] + (["--limit", "3"] if SMOKE else [])
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])

In [ ]:
import json, pandas as pd
OUT = ARTIFACTS / "assoc" / ("stage0_timing_v2" + ("_smoke" if SMOKE else ""))
s = json.loads((OUT / "summary.json").read_text())
print(json.dumps(s["meta"], indent=1))
for k in ["clips", "clips_error", "clip_errors", "frame_status", "clips_all_frames_identified_frac",
          "hypothesis_agreement_on_identified", "identified_abs_mf4_residual_s", "pts_crosscheck_ok"]:
    print(k, ":", s["summary"][k])
print("files:", sorted(p.name for p in OUT.iterdir()))